In [1]:
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
import pickle

# Define the DATA_DIRECTORY and FIGURE_DIRECTORY
HERE = os. getcwd()
DATA_DIRECTORY = os.path.abspath(os.path.join(HERE, os.pardir, 'data'))
FIGURE_DIRECTORY = os.path.abspath(os.path.join(HERE, os.pardir, 'figures'))



In [2]:
id_rank_code_dict_path = os.path.join(DATA_DIRECTORY, 'external', 'id_rank_code_dict.pkl')

with open(id_rank_code_dict_path, "rb") as fp:
    rank_code_dictionary = pickle.load(fp)
    rank_code_dictionary['methane_grams_per_day']="Phenotype"


In [3]:
def feature_dropna(sample_matrix):    
    # fill NA by "0" and filter out columns with all zeros
    #print("Number of features from Silva: "+str(len(list(sample_matrix))))
    sample_matrix_woNA = sample_matrix.dropna(axis=1, how = 'all')
    sample_matrix_woNA_woZero = sample_matrix_woNA.loc[:, (sample_matrix_woNA != 0).any(axis=0)]
    #print("Number of features after '0' dropping: "+str(len(list(sample_matrix_woNA_woZero))))
    features = list(sample_matrix_woNA_woZero)
    return sample_matrix_woNA_woZero, features


def low_abundance_filtering(count_df, min_frequency, min_percent):
    for feature in count_df:
        count = 0
        for v in count_df[feature]:
            if v >= min_frequency:
                count += 1  
        if count/len(count_df) < min_percent:
            count_df = count_df.drop(feature, inplace=False, axis=1)
    return count_df


def relative_abundance(sample_matrix):
    
    # Add 1 pseudo count
    cols = list(sample_matrix.columns)
    sample_matrix[cols]+=1
    
    # relative abundace sample-wise, multiplied by 1000,000
    sample_matrix_relative = sample_matrix.div(sample_matrix.sum(axis=1), axis=0)
    sample_matrix_relative_float = sample_matrix_relative.astype(float)
    
    # retrieve the y

    return sample_matrix_relative_float


def reverse_non_unique_mapping(d):
    dinv = {}
    for k, v in d.items():
        if v in dinv:
            dinv[v].append(str(k))
        else:
            dinv[v] = [str(k)]
    return dinv


def taxonomic_level_pick(tax_otu_dict, tax_level, df):
    picked = df[df.columns.intersection(tax_otu_dict[tax_level])]
    print(f'The sum of OTUs at {tax_level} level: {len(picked.columns)}')
    return picked


def z_score(df): 
    # create a scaler object
    std_scaler = StandardScaler()
    # fit and transform the data
    df_std = pd.DataFrame(std_scaler.fit_transform(df), columns=df.columns)
    df_std.index = df.index
    print(f'The sum of OTUs in the end: {len(df_std.columns)}')
    return df_std
    

# 1. Dataset Difford

In [4]:
# Retrieve the ids of samples that has microbiome data

difford_taxa_path = os.path.join(DATA_DIRECTORY, 'interim','Difford2018', 'taxonomy', 'all_taxa_emission_reads_number.pkl' )
difford_tax_df = pd.read_pickle(difford_taxa_path)
difford_tax_df= difford_tax_df.sort_index()
difford_ids = difford_tax_df.index.values.tolist()

# remove the phenotype column for OTU table
cols = list(difford_tax_df.columns)
cols.remove('methane_grams_per_day')
difford_tax_without_phenotype = difford_tax_df[cols]

# loading the corresponding metadata data
difford_metadata_path = os.path.join(DATA_DIRECTORY, 'raw','Difford2018', 'emission_metadata.tsv' )
difford_metadata_df = pd.read_csv(difford_metadata_path, sep=' ', header=0)
difford_metadata_df= difford_metadata_df.sort_values(by=['sample_id'])

difford_metadata_df = difford_metadata_df.rename({"sample_id":"metadata_sample_id"}, axis=1)
difford_metadata_df = difford_metadata_df.set_index('metadata_sample_id')
difford_metadata_df = difford_metadata_df[difford_metadata_df.index.isin(difford_ids)]


In [5]:
difford_X, difford_y =  difford_tax_without_phenotype, difford_tax_df.methane_grams_per_day

In [6]:
# Drop features that are 0 across all samples
difford_X, difford_X_features = feature_dropna(difford_X)

# Filter out features that are present in less than 50% samples
difford_X_filteredLow = low_abundance_filtering(difford_X, 1, 0.5)

# Relative abundance transformation
difford_X_filteredLow_relative = relative_abundance(difford_X_filteredLow)

# Genus-level OTU picking
tax_otu_dict = reverse_non_unique_mapping(rank_code_dictionary)
difford_X_filteredLow_relative_genus = taxonomic_level_pick(tax_otu_dict, "G", difford_X_filteredLow_relative)

# z-score standardization
difford_X_filteredLow_relative_genus_standardized = z_score(difford_X_filteredLow_relative_genus)
difford_X_filteredLow_relative_genus_standardized.to_excel(os.path.join(DATA_DIRECTORY, 'interim','Difford2018', 'taxonomy', 'difford_X_filteredLow_relative_genus_standardized.xlsx'),index_label='metadata_sample_id')

The sum of OTUs at G level: 429
The sum of OTUs in the end: 429


# 2. Dataset Wallace

In [7]:
# Retrieve the OTU table
wallace_taxa_path = os.path.join(DATA_DIRECTORY, 'interim','Wallace2019', 'taxonomy','all_dict_reads_table.tsv')
wallace_taxa_df = pd.read_csv(wallace_taxa_path, sep='\t', index_col=0, low_memory=False)
wallace_y = wallace_taxa_df['methane_grams_per_day']
wallace_X = wallace_taxa_df.drop('methane_grams_per_day', axis=1)


In [14]:
# Drop features that are 0 across all samples
wallace_X, wallace_X_features = feature_dropna(wallace_X)

# Filter out features that are present in less than 50% samples
wallace_X_filteredLow = low_abundance_filtering(wallace_X, 1, 0.5)

# Relative abundance transformation
wallace_X_filteredLow_relative = relative_abundance(wallace_X_filteredLow)

# Genus-level OTU picking
tax_otu_dict = reverse_non_unique_mapping(rank_code_dictionary)
wallace_X_filteredLow_relative_genus = taxonomic_level_pick(tax_otu_dict, "G", wallace_X_filteredLow_relative)

# z-score standardization
wallace_X_filteredLow_relative_genus_standardized = z_score(wallace_X_filteredLow_relative_genus)
wallace_X_filteredLow_relative_genus_standardized.to_excel(os.path.join(DATA_DIRECTORY, 'interim','Wallace2019', 'taxonomy', 'wallace_X_filteredLow_relative_genus_standardized.xlsx'),index_label='metadata_sample_id')

The sum of OTUs at G level: 163
The sum of OTUs in the end: 163


# 3. Dataset Merged


In [10]:
# Shared genus-level OTU picking

def intersection(lst1, lst2):
    print(f'Shared bacterial and archaeal genus-level OTUs: {len(list(set(lst1) & set(lst2)))}')
    return list(set(lst1) & set(lst2))
    


difford_X_genus = taxonomic_level_pick(tax_otu_dict, "G", difford_X)
wallace_X_genus = taxonomic_level_pick(tax_otu_dict, "G", wallace_X)

shared_BA_features = intersection(difford_X_genus, wallace_X_genus)

The sum of OTUs at G level: 2677
The sum of OTUs at G level: 1152
Shared bacterial and archaeal genus-level OTUs: 1011


In [11]:
difford_X_genus_shared = difford_X_genus[shared_BA_features]
wallace_X_genus_shared = wallace_X_genus[shared_BA_features]
X_genus_shared = pd.concat([difford_X_genus_shared, wallace_X_genus_shared])

In [12]:
# Filter out features that are present in less than 50% samples
X_genus_shared_filteredLow = low_abundance_filtering(X_genus_shared, 1, 0.5)

# Relative abundance transformation
X_genus_shared_filteredLow_relative = relative_abundance(X_genus_shared_filteredLow)


# z-score standardization
X_genus_shared_filteredLow_relative_standardized = z_score(X_genus_shared_filteredLow_relative)

The sum of OTUs in the end: 182


In [13]:
X_genus_shared_filteredLow_relative_standardized.to_excel(os.path.join(DATA_DIRECTORY, 'interim','Merged', 'taxonomy', 'X_genus_shared_filteredLow_relative_standardized.xlsx'),index_label='metadata_sample_id')